In [2]:
from IPython.core.display import Markdown
from agents import ModelSettings, WebSearchTool, Agent, Runner, function_tool
from dotenv import load_dotenv
from google.adk.cli.utils.common import BaseModel
from pydantic import Field

load_dotenv(override=True)

INSTRUCTION = """
You are a research assistant. Given a search term, you search the web for that term and produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""

task = "Most popular AI agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]


In [3]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTION, model_settings=settings, tools=tools)

In [5]:

result = await Runner.run(search_agent, task)

In [7]:
display(Markdown(result.final_output))

In 2026, the most popular AI agent frameworks are **LangChain/LangGraph, OpenAI Agents SDK, CrewAI, AutoGen, LlamaIndex, Google ADK, and Pydantic AI**. In Stack Overflow’s 2026 developer survey, LangChain led intended use at 30.2%, followed by OpenAI Agents SDK (23.8%), LangGraph (18.8%), Llama Stack (17.4%), and Google ADK (14.0%). CrewAI (5.5%) and AutoGen (4.5%) remained well-known but had lower reported use. ([survey.stackoverflow.co](https://survey.stackoverflow.co/2026/ai/data/ai-orch?utm_source=openai))

Popularity varies by metric: GitHub stars favor **LangChain, AutoGen, CrewAI, and LlamaIndex**, while package-download data places **LangChain, LangGraph, OpenAI Agents SDK, Google ADK, and Pydantic AI** near the top. For practical selection, LangGraph is commonly associated with structured, stateful production workflows; CrewAI with role-based multi-agent teams; LlamaIndex with data and RAG-heavy agents; OpenAI Agents SDK and Google ADK with vendor-native development; and Pydantic AI with lightweight, typed Python applications. ([boringparts.net](https://boringparts.net/en/ai-agent-framework-comparison-table-github-stars-pypi-downloads/?utm_source=openai))

In [4]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="The reason why this search result was included in the summary.")
    query: str = Field(description="The search query used to obtain this result.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [5]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'The reason why this search result was included in the summary.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search query used to obtain this result.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [6]:
INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output 5 terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model="gpt-5.4-mini", output_type=WebSearchPlan)

In [7]:
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find current rankings and market surveys on AI agent frameworks in 2026.', query='most popular AI agent frameworks 2026 ranking'), WebSearchItem(reason='Identify open-source and commercial frameworks commonly discussed as leading options.', query='best AI agent frameworks 2026 LangChain AutoGen CrewAI LangGraph comparison'), WebSearchItem(reason='Check adoption signals such as GitHub stars, downloads, and community activity.', query='AI agent frameworks GitHub stars 2026'), WebSearchItem(reason='Look for industry reports, benchmarks, or roundups from reputable sources.', query='AI agent framework trends report 2026'), WebSearchItem(reason='Capture emerging frameworks that may have gained popularity recently.', query='new AI agent frameworks 2026')])

In [8]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim
for 5 -10 pages of content, at least 1000 words.
"""

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model="gpt-5.4-mini", output_type=ReportData)



In [10]:
from messeger import send_email
from agents import function_tool


@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    :param subject:  the subject of the email
    :param text_body:  the body of the email
    :param html_body: htmled version of the body
    """

    send_email(subject, text_body, html_body)
    return "Email sent successfully."

In [12]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'the subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'the body of the email',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'htmled version of the body',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [14]:
INSTRUCTIONS = """
You are provided with a detailed report. use your tool to send an email, converting the report into c clean,
well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model="gpt-5.4-mini")

In [15]:
import asyncio
async def run_researches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print(f"Finished {len(results)} searches")
    return results

async def search(item: WebSearchItem):
    print(f"Searching for {item.query} because {item.reason}")
    result = await Runner.run(search_agent, item.query)
    return result.final_output

In [16]:
async def write_report(query: str, search_results: list[str]):
    print(f"Thinking about report...")
    input_message = f"Original query: {query}\n\nResearch: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    return result.final_output

async def send_report_email(report: ReportData):
    print(f"Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print(f"Email sent successfully")
    return report

In [17]:
query = "MOst popular AI agent frameworks in 2026"
from agents import trace
with trace("Research trace"):
    print(f"Starting research for query: {query}")
    search_results = await run_researches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)
    print("Research complete!")

Starting research for query: MOst popular AI agent frameworks in 2026
Planning searches...
Will perform 5 searches
Searching for most popular AI agent frameworks 2026 because Find current rankings, comparisons, and market mentions of leading AI agent frameworks in 2026.
Searching for top AI agent frameworks developer survey 2026 because Identify framework adoption signals and community interest across developer sources.
Searching for best open source AI agent frameworks 2026 because Compare open-source agent frameworks and their usage in production.
Searching for AI agent framework comparison 2026 LangChain CrewAI AutoGen because See which frameworks are being recommended in recent technical reviews and benchmark articles.
Searching for GitHub stars AI agent frameworks 2026 because Check GitHub popularity and activity to infer popularity among agent frameworks.
Finished 5 searches
Thinking about report...
Writing email...
Email sent successfully
Research complete!
